## 结构化输出

## model的结构化输出策略

In [2]:
# model
from rich import print as rprint
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="deepseek-v4-flash", # 模型名称
    extra_body={
        "thinking": {"type": "disabled"} # 关闭思考模式，有些模型思考模式不支持结构化输出
    }
)


In [6]:
from pydantic import BaseModel, Field
from typing import Optional

class MovieModel(BaseModel):
    name: str = Field(description="电影名称")
    year: Optional[int] = Field(default="null", description="发行年份")
    director: Optional[str] = Field(default="null", description="导演")
    rating: Optional[float] = Field(default="null", description="评分")

structured_model = model.with_structured_output(MovieModel)

response = structured_model.invoke("我最近看了一部诺兰的电影，叫星际穿越，主角演技很好叫马修")
# response = structured_model.invoke("介绍一下星际穿越")

rprint(response)


MovieModel(name='星际穿越', year=2014, director='克里斯托弗·诺兰', rating='null')

## agent结构化输出策略

In [3]:
# Pydantic
from pydantic import BaseModel, Field

class ContractInfo(BaseModel):
    name: str = Field(description="用户姓名")
    age: int = Field(description="用户年龄")
    address: str = Field(description="用户地址")

In [4]:
# agent
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

agent = create_agent(
    name="structured_output_agent", # 一般用于Multi-Agent场景
    model = model,
    system_prompt="你是一个话少的助手",
    response_format = ToolStrategy(schema=ContractInfo)
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": "有一个客户名字叫Alice，她是个18岁的女生，她和父母住在一起，在北京的东城区"},
    ]
})

rprint(response)

D:\develop\project\agent-learning-notes\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{
    'messages': [
        HumanMessage(
            content='有一个客户名字叫Alice，她是个18岁的女生，她和父母住在一起，在北京的东城区',
            additional_kwargs={},
            response_metadata={},
            id='1431c58e-2933-4470-9695-4e66b80d67bc'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 70,
                    'prompt_tokens': 344,
                    'total_tokens': 414,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 344
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '487fe451-2def-45fc-bc27-c93614c44f3f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            name='structured_output_agent',
            id='lc_run--019f8e2b-c272-7072-ae78-c70559bdd137-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': 'Alice', 'age': 18, 'address': '北京市东城区'},
                    'id': 'call_00_EgVHHpEZ3YCUAFzmp0Zx2012',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 344,
                'output_tokens': 70,
                'total_tokens': 414,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='Alice' age=18 address='北京市东城区'",
            name='ContractInfo',
            id='b8aae1a9-b875-495e-8138-4c124ab09785',
            tool_call_id='call_00_EgVHHpEZ3YCUAFzmp0Zx2012'
        )
    ],
    'structured_response': ContractInfo(name='Alice', age=18, address='北京市东城区')
}